# DA6401 Assignment 1 — W&B Demo Notebook

This notebook demonstrates how to:
1. Train the NumPy MLP and log metrics to Weights & Biases
2. Run a hyperparameter sweep
3. Visualise results in W&B

**Note:** All mathematical operations use NumPy only. No PyTorch/TensorFlow.

In [ ]:
# Install dependencies if running in Colab
# !pip install numpy keras scikit-learn matplotlib wandb

In [ ]:
import sys
import os
import numpy as np
import wandb
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from ann.neural_network import NeuralNetwork
from ann.optimizers import get_optimizer
from utils.data_loader import load_data

print('Imports successful!')

## 1. Load Data

In [ ]:
# Load Fashion-MNIST
x_train, x_val, x_test, y_train, y_val, y_test = load_data('fashion_mnist')

print(f'Train shape: {x_train.shape}')
print(f'Val shape:   {x_val.shape}')
print(f'Test shape:  {x_test.shape}')

## 2. Train with W&B Logging

In [ ]:
# Configuration
config = {
    'dataset':       'fashion_mnist',
    'epochs':        15,
    'batch_size':    64,
    'loss':          'cross_entropy',
    'optimizer':     'adam',
    'learning_rate': 0.001,
    'weight_decay':  0.0,
    'hidden_sizes':  [128, 128],
    'activation':    'relu',
    'weight_init':   'xavier',
}

# Initialise W&B run
wandb.init(
    project='da6401-assignment1',
    name='notebook-demo',
    config=config
)

# Build model and optimizer
model = NeuralNetwork(
    input_size=784,
    hidden_sizes=config['hidden_sizes'],
    output_size=10,
    activation=config['activation'],
    weight_init=config['weight_init'],
    loss=config['loss'],
)

optimizer = get_optimizer(
    name=config['optimizer'],
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
)

# Training loop
best_val_acc = 0.0
num_batches  = int(np.ceil(x_train.shape[0] / config['batch_size']))

for epoch in range(1, config['epochs'] + 1):
    idx = np.random.permutation(x_train.shape[0])
    x_tr, y_tr = x_train[idx], y_train[idx]

    epoch_loss, correct = 0.0, 0
    for b in range(num_batches):
        xb = x_tr[b * config['batch_size']: (b+1) * config['batch_size']]
        yb = y_tr[b * config['batch_size']: (b+1) * config['batch_size']]

        logits = model.forward(xb)
        loss   = model.compute_loss(logits, yb)
        model.backward()
        optimizer.step(model.layers)

        epoch_loss += loss * xb.shape[0]
        correct    += (np.argmax(logits, axis=1) == yb).sum()

    train_loss = epoch_loss / x_train.shape[0]
    train_acc  = correct   / x_train.shape[0]
    val_logits = model.forward(x_val)
    val_acc    = (np.argmax(val_logits, axis=1) == y_val).mean()
    val_loss   = model.compute_loss(val_logits, y_val)

    print(f'Epoch {epoch:2d} | loss={train_loss:.4f} acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}')

    wandb.log({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        'val_loss': val_loss,
        'val_accuracy': val_acc,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc

wandb.finish()
print(f'\nBest val accuracy: {best_val_acc:.4f}')

## 3. Run W&B Sweep
To run a sweep from command line:
```bash
wandb sweep sweep.yaml
wandb agent <entity>/<project>/<sweep_id> --count 100
```

In [ ]:
# Example: launch sweep programmatically
import yaml

with open('../sweep.yaml') as f:
    sweep_config = yaml.safe_load(f)

sweep_id = wandb.sweep(sweep_config, project='da6401-assignment1')
print(f'Sweep ID: {sweep_id}')
print('Run: wandb agent <entity>/da6401-assignment1/' + sweep_id)